# 🤖 Notebook 02 — Model Training & Evaluation
## BI Predictive Analytics Platform · Multi-Model ML Pipeline

---

> **Author:** Mohammed Farhan Khan  
> **Stack:** Scikit-learn · XGBoost · LightGBM · Plotly

---

## 🎯 Objective

Train, tune, and rigorously evaluate **5 classification models** to predict customer churn.  
We use **stratified 5-fold cross-validation** to ensure robust, unbiased performance estimates.  
Every metric is reported with mean ± std to capture model stability.

## 📋 Table of Contents

1. [Feature Engineering & Preprocessing](#1-preprocessing)
2. [Model Training with Cross-Validation](#2-training)
3. [Performance Comparison](#3-comparison)
4. [ROC Curves & AUC](#4-roc)
5. [Confusion Matrix Analysis](#5-confusion)
6. [Feature Importance](#6-importance)
7. [Model Selection Decision](#7-decision)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, roc_auc_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

DARK = 'plotly_dark'
print('✅ Imports ready')

## 1. Feature Engineering & Preprocessing <a id='1-preprocessing'></a>

In [ ]:
from src.data_pipeline.data_loader import DataLoader
from src.data_pipeline.preprocessor import Preprocessor

# Load data
loader = DataLoader()
df = loader.generate_sample_churn_data(n=5000, seed=42)
print(f'Raw dataset: {df.shape}')

# Full preprocessing pipeline
pp = Preprocessor(target_col='churn', test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = pp.fit_transform(df)

print(f'\n✅ Preprocessing complete:')
print(f'   X_train: {X_train.shape} | y_train churn rate: {y_train.mean():.1%}')
print(f'   X_test:  {X_test.shape}  | y_test  churn rate: {y_test.mean():.1%}')
print(f'   Features engineered: {len(pp.feature_names_)} total features')
print(f'\n📐 Engineered features added:')
new_feats = [f for f in pp.feature_names_ if f in ['avg_monthly_spend','support_call_rate','engagement_score','charge_variance']]
for f in new_feats:
    print(f'   + {f}')

## 2. Model Training with Cross-Validation <a id='2-training'></a>

In [ ]:
from src.models.classification_models import ModelSuite

suite = ModelSuite(cv_folds=5, save_dir='../models/saved')

print('🏋️  Training all models with 5-fold Stratified CV...')
print('─' * 55)
suite.train_all(X_train, y_train)

print('\n📊 CROSS-VALIDATION RESULTS (on training set):')
print('─' * 55)
rows = []
for name, results in suite.cv_results.items():
    row = {'Model': name.replace('_', ' ').title()}
    for metric, vals in results.items():
        row[metric.upper()] = f"{vals['mean']:.4f} ± {vals['std']:.4f}"
    rows.append(row)

cv_df = pd.DataFrame(rows).set_index('Model')
cv_df.style.set_caption('5-Fold CV Results')

In [ ]:
suite.save_all()
best = suite.best_model('f1')
print(f'✅ All models saved to models/saved/')
print(f'🏆 Best model by F1-Score: {best.replace("_", " ").title()}')

## 3. Performance Comparison <a id='3-comparison'></a>

In [ ]:
from src.models.evaluator import Evaluator

evaluator = Evaluator(output_dir='../reports/figures')
summary = evaluator.evaluate(suite.trained_models, X_test, y_test, feature_names=pp.feature_names_)

print('\n🏆 MODEL LEADERBOARD (Test Set):')
print('─' * 60)
summary['model_display'] = summary['model'].str.replace('_', ' ').str.title()
print(summary[['model_display', 'f1_score', 'roc_auc', 'accuracy', 'avg_precision']].to_string(index=False))

# Styled leaderboard
display_df = summary[['model_display', 'f1_score', 'roc_auc', 'accuracy', 'avg_precision']].copy()
display_df.columns = ['Model', 'F1-Score', 'ROC-AUC', 'Accuracy', 'Avg Precision']
display_df.style \
    .background_gradient(cmap='Blues', subset=['F1-Score', 'ROC-AUC', 'Accuracy']) \
    .highlight_max(subset=['F1-Score', 'ROC-AUC'], color='#16A34A33') \
    .format(precision=4) \
    .set_caption('🏆 Model Leaderboard — Test Set Performance')

In [ ]:
metrics = ['f1_score', 'roc_auc', 'accuracy', 'avg_precision']
metric_labels = ['F1-Score', 'ROC-AUC', 'Accuracy', 'Avg Precision']
colors = ['#1A56DB', '#7C3AED', '#06B6D4', '#22C55E']

fig = go.Figure()
for metric, label, color in zip(metrics, metric_labels, colors):
    fig.add_trace(go.Bar(
        name=label,
        x=summary['model'].str.replace('_', ' ').str.title(),
        y=summary[metric],
        marker_color=color,
        text=[f'{v:.3f}' for v in summary[metric]],
        textposition='outside'
    ))

fig.update_layout(
    barmode='group', template=DARK, height=460,
    title='📊 Model Performance Comparison — All Metrics',
    title_font=dict(size=17, color='#E2E8F0'),
    yaxis=dict(range=[0, 1.08], title='Score'),
    xaxis_title='Model',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

## 4. ROC Curves & AUC <a id='4-roc'></a>

In [ ]:
model_colors = {
    'logistic_regression': '#06B6D4',
    'random_forest':        '#22C55E',
    'gradient_boosting':    '#F59E0B',
    'xgboost':              '#EF4444',
    'lightgbm':             '#7C3AED',
}

fig = go.Figure()
fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1,
              line=dict(color='#475569', width=1.5, dash='dash'))

for name, model in suite.trained_models.items():
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        color = model_colors.get(name, '#1A56DB')
        fig.add_trace(go.Scatter(
            x=fpr, y=tpr, mode='lines',
            name=f'{name.replace("_", " ").title()} (AUC = {auc:.4f})',
            line=dict(color=color, width=2.5)
        ))

fig.update_layout(
    template=DARK, height=500,
    title='📈 ROC Curves — All Models',
    title_font=dict(size=17, color='#E2E8F0'),
    xaxis=dict(title='False Positive Rate', range=[-0.02, 1.02]),
    yaxis=dict(title='True Positive Rate', range=[-0.02, 1.02]),
    legend=dict(yanchor='bottom', y=0.01, xanchor='right', x=0.99)
)
fig.show()

## 5. Confusion Matrix Analysis <a id='5-confusion'></a>

In [ ]:
import plotly.figure_factory as ff

model_names = list(suite.trained_models.keys())
n = len(model_names)
cols = min(3, n)
rows = (n + cols - 1) // cols

fig = make_subplots(rows=rows, cols=cols,
    subplot_titles=[m.replace('_', ' ').title() for m in model_names])

for i, (name, model) in enumerate(suite.trained_models.items()):
    r, c = i // cols + 1, i % cols + 1
    cm = confusion_matrix(y_test, model.predict(X_test))
    fig.add_trace(go.Heatmap(
        z=cm, colorscale='Blues',
        text=cm, texttemplate='%{text}', textfont_size=16,
        x=['Predicted: No Churn', 'Predicted: Churn'],
        y=['Actual: No Churn', 'Actual: Churn'],
        showscale=False
    ), row=r, col=c)

fig.update_layout(template=DARK, height=rows * 250 + 80,
    title='🎯 Confusion Matrices — All Models',
    title_font=dict(size=17, color='#E2E8F0'))
fig.show()

# Detailed report for best model
best_model = suite.trained_models[best]
print(f'\n📋 Classification Report — {best.replace("_", " ").title()} (Best Model):')
print('─' * 60)
print(classification_report(y_test, best_model.predict(X_test),
    target_names=['Retained', 'Churned']))

## 6. Feature Importance <a id='6-importance'></a>

In [ ]:
# Feature importance for all tree-based models
tree_models = {k: v for k, v in suite.trained_models.items() if hasattr(v, 'feature_importances_')}
feature_names = pp.feature_names_

fig = make_subplots(rows=1, cols=len(tree_models),
    subplot_titles=[m.replace('_', ' ').title() for m in tree_models])

colors_cycle = ['#1A56DB', '#22C55E', '#F59E0B', '#EF4444', '#7C3AED']

for idx, (name, model) in enumerate(tree_models.items()):
    importances = model.feature_importances_
    top_n = 12
    indices = np.argsort(importances)[-top_n:]
    feats = [feature_names[i].replace('_', ' ').title() for i in indices]
    vals = importances[indices]
    color = colors_cycle[idx % len(colors_cycle)]
    
    fig.add_trace(go.Bar(
        y=feats, x=vals, orientation='h',
        marker_color=color, name=name.replace('_', ' ').title(),
        text=[f'{v:.3f}' for v in vals], textposition='outside',
        showlegend=False
    ), row=1, col=idx+1)

fig.update_layout(
    template=DARK, height=500,
    title='🔍 Top 12 Feature Importances — Tree-Based Models',
    title_font=dict(size=17, color='#E2E8F0')
)
fig.show()

print('\n💡 INSIGHT: Tenure, charges and satisfaction consistently rank highest.')
print('   Engineered features (avg_monthly_spend, support_call_rate) also appear — validating our FE step.')

## 7. Model Selection Decision <a id='7-decision'></a>

In [ ]:
print('=' * 65)
print('  MODEL SELECTION DECISION')
print('=' * 65)

best_row = summary.loc[summary['model'] == best].iloc[0]
print(f'\n  🏆 Selected Model : {best.replace("_", " ").title()}')
print(f'  📊 F1-Score        : {best_row["f1_score"]:.4f}')
print(f'  📈 ROC-AUC         : {best_row["roc_auc"]:.4f}')
print(f'  ✅ Accuracy        : {best_row["accuracy"]:.4f}')
print(f'  🎯 Avg Precision   : {best_row["avg_precision"]:.4f}')

print('\n  RATIONALE:')
print('  - Highest F1-Score → best balance of precision & recall')
print('  - High ROC-AUC → strong rank-ordering of churn probability')
print('  - Robust to class imbalance via class_weight="balanced"')
print('  - Feature importances are interpretable for business stakeholders')
print()
print('=' * 65)
print('\n→ Proceed to Notebook 03 for Business Impact & ROI Analysis')

---

### 🔜 Next Steps
➡️ [Notebook 03 — Business Impact & ROI Analysis](./03_Business_Impact_and_ROI.ipynb)  
➡️ [Back to Notebook 01 — EDA](./01_EDA_Customer_Churn.ipynb)